In [1]:
import os
import json
import pandas as pd
from datasets import Dataset
from dotenv import load_dotenv

In [2]:
# .env 파일에서 환경변수 로드
load_dotenv()

True

In [3]:
# 허깅페이스 토큰 및 데이터셋 주소 수정포인트 본인 토큰 맞는지 확인 필요
hf_token = os.getenv('HF_TOKEN')
hf_repo = os.getenv('HF_DATASET_REPO', 'hyokwan/health_essential_knowledge')

In [5]:
parent_dir = './dataset/01. 필수의학'  # 예: 'dataset/전문의료'

# __file__이 없는 환경(Jupyter 등)에서도 동작하도록 처리 (현재파일 위치 설정)
try:
    script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    script_dir = os.getcwd()

# 처리할 폴더 리스트 (parent_dir 내 모든 하위 폴더 자동 탐색)
dataset_root = os.path.join(script_dir, parent_dir)

dirs = os.listdir(dataset_root)
data_folders = []
for i in range(len(dirs)):
    d = dirs[i]
    if os.path.isdir(os.path.join(dataset_root, d)):
        data_folders.append(d)

all_rows = []

In [6]:
for i in range(0, len(data_folders)):
    folder = data_folders[i]
    data_dir = os.path.join(dataset_root, folder)

    if not os.path.exists(data_dir):
        continue

    if '_' in folder:
        input_val = folder.split('_', 1)[1]
    else:
        input_val = folder

    files = os.listdir(data_dir)

    for j in range(0, len(files)):
        file = files[j]

        if not file.endswith('.json'):
            continue

        file_path = os.path.join(data_dir, file)

        try:
            with open(file_path, 'r', encoding='utf-8-sig') as f:
                data = json.load(f)
        except UnicodeDecodeError:
            with open(file_path, 'r', encoding='ms949') as f:
                data = json.load(f)

        # 리스트인 경우
        if type(data) == list:
            for k in range(0, len(data)):
                item = data[k]

                all_rows.append({
                    'instruction': item['question'],
                    'input': input_val,
                    'output': item['answer']
                })

        # dict 한 개인 경우
        else:
            all_rows.append({
                'instruction': data['question'],
                'input': input_val,
                'output': data['answer']
            })
# 데이터프레임 생성
pdf = pd.DataFrame(all_rows)

In [7]:
pdf

,instruction,input,output
0,23세 여자가 3개월 전부터 기침을 한다며 내원했다. 기침은 밤에 누워 자려고 할 ...,내과,4) 메타콜린 기관지유발검사
1,35세 남자가 6개월 동안 지속된 기침으로 내원했다. 기침 시 숨이 차며 가래는 없...,내과,2) 글루코코르티코이드 치료
2,다음 중 재생불량빈혈(Aplastic Anemia)의 중증 기준에 해당하는 것은 무...,내과,"2) 절대 호중구 수 500/mm³ 미만, 혈소판 20,000/mm³ 미만, 교정 ..."
3,62세 여자가 양쪽 다리에 멍이 들어 내원했다. 평소에 코피가 잘 나는 편이라고 하...,내과,1) 삼중면역억제치료
4,52세 여자가 3일 전부터 다리에 멍이 들고 손목에 붓기와 통증을 느껴 내원했다. ...,내과,3) 프로타민설페이트
...,...,...,...
15355,"50세 남자가 화재 현장에서 구조된 후 두통, 어지러움, 의식 저하를 호소하며 내원...",응급의학과,2) 100% 산소 공급
15356,외상성 기흉이 의심되는 응급 환자에서 가장 적절한 초기 진단 방법은 무엇인가? \...,응급의학과,3) 흉부 초음파 검사
15357,"심정지 환자를 발견했을 때, 심폐소생술을 시작하기 전에 가장 먼저 확인해야 할 사항...",응급의학과,4) 주변 환경의 안전 여부
15358,58세 여성이 외래에서 심정지로 응급 처치를 받게 되었다. 그녀는 즉시 심폐소생술(...,응급의학과,1) 피부 외상 및 멍


In [13]:
# Huggingface Datasets 객체로 변환
hf_dataset = Dataset.from_pandas(pdf)

# 허깅페이스 허브에 업로드
print(f"허깅페이스 허브({hf_repo})에 업로드 시도...")
hf_dataset.push_to_hub(hf_repo, token=hf_token)
print("허깅페이스 업로드 완료!")

허깅페이스 허브(Mazelone/health_essential_knowledge)에 업로드 시도...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/16 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

C:\Users\HP\.llmall\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\datasets--Mazelone--health_essential_knowledge. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


허깅페이스 업로드 완료!
